# Homework 1 Solution

This notebook shows one possible solution path for Homework 1.

The repository includes a lightweight WDI subset for demonstration. Replace it with the full WDI file if your assignment requires a country that is not included here.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "homework_01"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

# Q1. Filter the data for the country you need.

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_DIR / "wdi" / "WDI_course_subset.csv")

country_name = "United States"
df_country = df[df["Country Name"] == country_name]

df_country.head()

# Q2. How many rows and columns does the selected country dataset have?

In [ ]:
df_country.shape

# Q3. Reshape the data into panel format.

In [ ]:
panel_data = df_country.drop(columns="Indicator Code").melt(
    id_vars=["Country Name", "Country Code", "Indicator Name"],
    var_name="Year",
).pivot_table(
    values="value",
    index=["Country Name", "Country Code", "Year"],
    columns="Indicator Name",
).reset_index().rename_axis("", axis=1)

panel_data.head()

In [ ]:
# Recheck the data types.
panel_data.dtypes

# Convert the original string-like Year column to integer.
panel_data["Year"] = panel_data["Year"].astype(str).astype(int)
panel_data.dtypes

# Q4. Export the panel data.

In [ ]:
panel_data.to_csv(MODULE_OUTPUT_DIR / "Homework1_panel_data.csv", index=False)

# Q5. How many missing values does each variable have?

In [ ]:
isna_data = panel_data.isna().sum().sort_values(ascending=True)
isna_data

# Bonus / Q6. Select the 10 indicators with the fewest missing values and create a compact panel dataset.

This is only one possible answer.

In [ ]:
id_vars = ['Country Name', 'Country Code', 'Year']

top_10_features = (
    isna_data.drop(labels=id_vars, errors='ignore')
    .head(10)
    .index
    .tolist()
)

compact_panel = panel_data.loc[:,id_vars + top_10_features ]
compact_panel.head()
